# Microsoft Foundry · Agente com ferramenta OpenAPI (clima)

Laboratório do curso de **Microsoft Foundry** (Seção 4 · Aula 16): como transformar **qualquer API REST** em uma ferramenta de agente apenas entregando a sua **especificação OpenAPI**. O exemplo é um agente do clima que consulta a API pública **wttr.in**.

## Sumário

0. [Visão geral do projeto](#0.-Visão-geral-do-projeto)
1. [Instalação das bibliotecas](#1.-Instalação-das-bibliotecas)
2. [Variáveis de ambiente](#2.-Variáveis-de-ambiente)
3. [Conectando ao projeto do Foundry](#3.-Conectando-ao-projeto-do-Foundry)
4. [Montando a ferramenta OpenAPI](#4.-Montando-a-ferramenta-OpenAPI)
5. [Criando o agente do clima](#5.-Criando-o-agente-do-clima)
6. [Cliente OpenAI e conversa](#6.-Cliente-OpenAI-e-conversa)
7. [Perguntando o clima ao agente](#7.-Perguntando-o-clima-ao-agente)
8. [Verificando no portal do Foundry](#8.-Verificando-no-portal-do-Foundry)
9. [Resumo dos aprendizados](#9.-Resumo-dos-aprendizados)
10. [Limpeza e custos](#10.-Limpeza-e-custos)
11. [Interface com Streamlit](#11.-Interface-com-Streamlit)

---
## 0. Visão geral do projeto

### 0.1 Problema

Empresas têm centenas de APIs (estoque, pedidos, clima, CRM…). Integrar cada uma a um assistente de IA costuma exigir código específico para decidir **qual endpoint chamar**, **com quais parâmetros** e **como ler a resposta**.

### 0.2 Objetivo

- Usar a **especificação OpenAPI** (o "contrato" padrão de quase toda API) como ferramenta de um agente do **Foundry Agent Service**.
- Deixar que o **agente leia as descrições** e decida sozinho o endpoint e os parâmetros.
- Consultar o clima em tempo real pela API pública **wttr.in** e conferir no **tracing** a chamada feita.

### 0.3 Ferramentas e serviços da Azure

| Serviço / ferramenta | Papel no projeto |
| --- | --- |
| **Microsoft Foundry** (projeto) | Reúne modelo, agente, conversas e traces. |
| **Foundry Agent Service** | Hospeda o `weather-agent` e executa as chamadas HTTP da ferramenta. |
| **Ferramenta OpenAPI** (`type: "openapi"`) | Transforma a especificação `weather_openapi.json` em uma ferramenta que o agente sabe usar. |
| **Azure OpenAI** (deployment GPT-4) | Escolhe a operação, preenche os parâmetros e resume o JSON da API. |
| **Microsoft Entra ID** | Autenticação sem chave (`DefaultAzureCredential`). |
| API externa **wttr.in** | Serviço público de clima, sem autenticação (`auth: anonymous`). |
| `jsonref` | Lê a especificação resolvendo referências (`$ref`) internas do JSON. |

### 0.4 Como funciona (visão simples)

![Fluxo do agente com OpenAPI](fluxo_openapi.svg)

### 0.5 Arquitetura

![Arquitetura do projeto](arquitetura.svg)

### 0.6 A especificação `weather_openapi.json`

A especificação OpenAPI descreve a API para pessoas **e para o agente**:

| Parte da spec | Valor | Por que importa para o agente |
| --- | --- | --- |
| `openapi` | `3.1.0` | Versão do padrão. |
| `info.title` / `info.description` | *get weather data* / *"Retrieves current weather data for a location based on wttr.in."* | É o que o agente lê para decidir **se** usa a ferramenta. |
| `servers[0].url` | `https://wttr.in` | Endereço base da API. |
| `paths` → `/{location}` → `get` | *"Get weather information for a specific location"* | Ajuda a escolher **qual endpoint** chamar. |
| `operationId` | `GetCurrentWeather` | Nome da operação (aparece no tracing). |
| parâmetro `location` (path) | *"City or location to retrieve the weather for"* | O agente preenche com a cidade da pergunta. |
| parâmetro `format` (query) | *"Always use j1 value for this parameter"* | `j1` = resposta em **JSON**. |
| `responses` | `200` sucesso · `404` local não encontrado | O agente sabe o que esperar. |

> 💡 Quanto melhores o título e as descrições, melhor o agente escolhe a ferramenta e preenche os parâmetros. Para APIs protegidas, a spec e a ferramenta também aceitam autenticação (chave, OAuth/Entra ID…).

---
## 1. Instalação das bibliotecas

| Biblioteca | Para que serve |
| --- | --- |
| `azure-ai-projects==2.0.0b2` | **Foundry SDK v2**: cliente do projeto (`AIProjectClient`), agentes e ferramentas. Versão *beta*, fixada para o código funcionar igual ao da aula. |
| `openai==1.109.1` | Cliente OpenAI, usado para criar conversas e enviar mensagens ao agente (Responses API). |
| `python-dotenv` | Lê o arquivo `.env` com as configurações. |
| `azure-identity` | Autenticação na Azure sem chave no código (`DefaultAzureCredential`). |
| `jsonref` | Lê a especificação OpenAPI resolvendo os `$ref`. ⚠️ É importada no notebook, mas **não está** no `%pip install` abaixo: se ainda não estiver instalada, rode `%pip install jsonref`. |

> ⚠️ Versões fixadas (`==`) evitam surpresas: SDKs em *beta* mudam nomes de classes e métodos entre versões.

In [ ]:
# Instalando as bibliotecas no mesmo ambiente do kernel do notebook (%pip)
# azure-ai-projects = Foundry SDK | openai = conversa com o agente
# python-dotenv = lê o .env | azure-identity = login na Azure sem chave no código
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

---
## 2. Variáveis de ambiente

As configurações ficam no arquivo **`.env`** (fora do código):

| Variável | O que é | Onde encontrar |
| --- | --- | --- |
| `FOUNDRY_PROJECT_ENDPOINT` | Endereço (URL) do projeto Foundry. | Portal do Foundry → página **Início** (*Overview*) → *Project endpoint*. |
| `MODEL_DEPLOYMENT_NAME` | Nome do deployment do modelo de chat. | Portal do Foundry → **Modelos + endpoints**. |


```
FOUNDRY_PROJECT_ENDPOINT=https://<seu-recurso>.services.ai.azure.com/api/projects/<seu-projeto>
MODEL_DEPLOYMENT_NAME=<nome-do-deployment>

```

> 🔒 **Segurança:** o `.env` guarda apenas **endereços e nomes** (nenhuma chave), porque a autenticação é feita pelo Microsoft Entra ID (seção seguinte). Se algum dia ele passar a ter segredos (chaves de API, tokens), não deve ser publicado: nesse caso, adicione `.env` ao `.gitignore`.

In [ ]:
# Importando as bibliotecas e lendo as configurações do arquivo .env
# os.getenv busca cada valor pelo nome; se a variável não existir, retorna None
# jsonref = lê o JSON da especificação OpenAPI resolvendo referências ($ref)
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
)
import jsonref

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

---
## 3. Conectando ao projeto do Foundry

O **`AIProjectClient`** é a porta de entrada do Python para o projeto no Foundry: por ele criamos agentes, listamos conexões e obtemos o cliente OpenAI.

- **`endpoint`**: o endereço do projeto (do `.env`).
- **`credential`**: o **`DefaultAzureCredential`** reaproveita um login que já existe na máquina (Azure CLI `az login`, conta Azure do VS Code, *Managed Identity* quando roda na Azure…). Assim, **nenhuma chave fica escrita no código**: quem autentica é o **Microsoft Entra ID**.

> 💡 Para rodar localmente, basta executar `az login` antes. O usuário precisa ter permissão no projeto (por exemplo, a função **Azure AI User**).

In [ ]:
# Criando o cliente do projeto Foundry: endpoint do .env + login da máquina (Entra ID)
# A conexão é "preguiçosa": erros de login só aparecem na primeira chamada real
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

---
## 4. Montando a ferramenta OpenAPI

Lemos a especificação do arquivo e montamos a ferramenta como um **dicionário**:

| Chave | Valor | Significado |
| --- | --- | --- |
| `type` | `"openapi"` | Tipo da ferramenta. |
| `openapi.name` | `"weather"` | Nome da ferramenta (aparece no portal). |
| `openapi.spec` | conteúdo do `weather_openapi.json` | O "manual" da API. |
| `openapi.auth.type` | `"anonymous"` | A API é pública: não precisa de credencial. |

In [ ]:
# Lendo a especificação OpenAPI e montando a ferramenta "weather"
# auth anonymous = API pública, sem chave
with open("./weather_openapi.json", "r") as f:
        openapi_weather = jsonref.loads(f.read())

# Initialize agent OpenApi tool using the read in OpenAPI spec
weather_tool = {
        "type": "openapi",
        "openapi":{
            "name": "weather",
            "spec": openapi_weather,
            "auth": {
                "type": "anonymous"
            },
        }
}

---
## 5. Criando o agente do clima

| Campo | Valor |
| --- | --- |
| `agent_name` | `weather-agent` |
| `instructions` | `"You are a helpful Weather Agent that provides weather information using the provided OpenAPI Tool"` → *"Você é um agente do clima útil que fornece informações meteorológicas usando a ferramenta OpenAPI fornecida."* |
| `tools` | `[weather_tool]` |

In [ ]:
# Criando o agente do clima com a ferramenta OpenAPI na lista de tools
agent_name = "weather-agent"

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are a helpful Weather Agent that provides weather information using the provided OpenAPI Tool",
        tools=[weather_tool]
    ),
)

# printing the agent id
print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

---
## 6. Cliente OpenAI e conversa

Nesta célula fazemos duas coisas de uma vez:

1. `project_client.get_openai_client()`: cliente OpenAI já autenticado no projeto (Responses API).
2. `openai_client.conversations.create()`: cria a **conversa** que guarda o histórico (`conversation.id`).

In [ ]:
# Obtendo o cliente OpenAI já autenticado no projeto Foundry
# É ele que cria conversas e envia mensagens ao agente (Responses API)
# e criando a conversa que guarda o histórico do chat
# creating an openai client first
openai_client = project_client.get_openai_client()

# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

---
## 7. Perguntando o clima ao agente

Pergunta: *"Como está o tempo na cidade de Nova York hoje?"*

In [ ]:
# Pergunta que exige dado em tempo real (clima de hoje)
user_input = "What's the weather like in New York City today?"

| Parâmetro | Valor | Para que serve |
| --- | --- | --- |
| `conversation` | `conversation.id` | Grava a mensagem na conversa e envia o histórico ao modelo. |
| `extra_body` → `agent` | `{"name": "weather-agent", "type": "agent_reference"}` | Diz **qual agente** responde. Sem versão informada, usa **a mais recente**. |
| `input` | pergunta do usuário | O texto enviado ao agente. |

A resposta final em texto fica em **`response.output_text`**. Repare que a pergunta **não manda** o agente usar a ferramenta: é o próprio agente que analisa a intenção e decide.

In [ ]:
# Enviando a pergunta: o agente lê a spec, chama a API wttr.in e resume o JSON
response = openai_client.responses.create(
            conversation=conversation.id,
            extra_body = {
                "agent": {
                    "name": agent_name,
                    "type": "agent_reference"
                }
            },
            input = user_input
        )
print(f"Weather Agent: {response.output_text}")

### 7.1 O que aconteceu na demonstração da aula

| Informação | Valor retornado |
| --- | --- |
| Temperatura atual | 2 °C (sensação de -2 °C) |
| Condição | Névoa congelante |
| Umidade | 51% |
| Vento | Oés-sudoeste, 17 km/h |
| Visibilidade | 10 km |
| Máxima / mínima | 4 °C / -3 °C |
| Índice UV | 0 |
| Nascer / pôr do sol | 7h40 / 16h28 |

✅ Abrindo `https://wttr.in/New York` no navegador, os valores conferem com os do agente: ele realmente buscou os dados na API.

---
## 8. Verificando no portal do Foundry

| Onde | O que aparece |
| --- | --- |
| **Agentes** → `weather-agent` → **Ferramentas** | A definição OpenAPI anexada ao agente. |
| **Traces** → `conversation.id` | Uma **chamada de função remota** para a operação `GetCurrentWeather`, com os argumentos `location = New York` e `format = j1`. |
| Metadados | Cerca de **11.026 tokens** no total. |

> 💡 **Inflação de tokens:** a pergunta e a resposta são curtas, mas o **JSON inteiro da API** entra no contexto do modelo. Em produção, vale preferir endpoints que devolvam só o necessário (ou um formato mais enxuto) para controlar custo e latência.

---
## 9. Resumo dos aprendizados

- Qualquer API com **especificação OpenAPI** vira ferramenta de agente com `{"type": "openapi", "openapi": {name, spec, auth}}`.
- O agente usa **título, descrições e parâmetros** da spec para decidir se chama a API, qual endpoint e com quais valores.
- `auth: anonymous` serve para APIs públicas; APIs protegidas podem usar chave ou identidade.
- O **tracing** mostra a operação chamada (`GetCurrentWeather`) e os argumentos.
- Respostas grandes de API **inflam os tokens**: um ponto de atenção para custo.

---
## 10. Limpeza e custos

| Recurso | Gera custo? |
| --- | --- |
| Chamadas ao modelo (tokens) | Sim, **por uso**: só quando você conversa com o agente. Ferramentas que trazem muito texto (páginas, APIs, documentos) aumentam bastante os tokens. |
| API wttr.in | Gratuita e pública (sem custo na Azure). Em APIs reais, considere limites de uso e cobrança do provedor. |

Ao terminar os estudos:

- **Portal do Foundry → Agentes**: apagar as versões do agente que não serão mais usadas.
- Ao final do curso, apagar o **grupo de recursos** no `portal.azure.com` remove tudo de uma vez.

> 💡 Vale criar um **orçamento com alerta** em *Cost Management → Budgets* para ser avisado de qualquer gasto inesperado.

---
## 11. Interface com Streamlit

A célula abaixo gera o arquivo **`app.py`**: o mesmo agente, agora com uma interface de chat no navegador. Ela reaproveita o código do notebook:

| Função do `app.py` | Código do notebook reaproveitado |
| --- | --- |
| `criar_clientes()` | `AIProjectClient` (seção 3) e `get_openai_client()` (seção 6) |
| `nova_conversa() / perguntar()` | `conversations.create` (seção 6) e `responses.create` (seção 7) |
| `ler_operacoes()` | lê o mesmo `weather_openapi.json` da seção 4 para mostrar as operações na barra lateral |
| `ferramentas_usadas()` | lista as etapas do agente (chamadas de ferramenta) vistas no tracing |

Pontos importantes:

- O Streamlit **reexecuta o script inteiro** a cada interação. Por isso os clientes ficam em cache (`@st.cache_resource`) e o `conversation.id` e o histórico ficam no `st.session_state`.
- O agente **não é recriado** a cada mensagem: o app usa o agente já criado neste notebook (pelo nome, sempre na versão mais recente).
- Abaixo de cada resposta aparecem as **etapas** que o agente executou (ex.: a chamada da ferramenta OpenAPI).

**Como executar:**

```bash
pip install streamlit
az login
streamlit run app.py
```

O app abre em `http://localhost:8501` e usa o mesmo arquivo `.env` desta pasta.

In [ ]:
%%writefile app.py

import json
import os
from pathlib import Path

import streamlit as st
from azure.ai.projects import AIProjectClient
from azure.core.exceptions import ClientAuthenticationError, HttpResponseError
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from openai import APIError

# ------------------------------------------------------------------
# 1. Configurações (as mesmas do notebook)
# ------------------------------------------------------------------
load_dotenv()

FOUNDRY_PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("MODEL_DEPLOYMENT_NAME")
AGENT_NAME = "weather-agent"  # criado na seção 5 do notebook
SPEC_PATH = Path(__file__).parent / "weather_openapi.json"

st.set_page_config(page_title="Weather Agent · Microsoft Foundry", page_icon="🌦️", layout="wide")


# ------------------------------------------------------------------
# 2. Conexão com o projeto do Foundry
# ------------------------------------------------------------------
@st.cache_resource
def criar_clientes():
    project_client = AIProjectClient(
        endpoint=FOUNDRY_PROJECT_ENDPOINT,
        credential=DefaultAzureCredential(),
    )
    return project_client.get_openai_client()


# ------------------------------------------------------------------
# 3. Resumo da especificação OpenAPI (só para exibir na barra lateral)
# ------------------------------------------------------------------
@st.cache_data
def ler_operacoes():
    spec = json.loads(SPEC_PATH.read_text(encoding="utf-8"))
    operacoes = []
    for caminho, metodos in spec["paths"].items():
        for metodo, detalhes in metodos.items():
            operacoes.append(f"`{metodo.upper()} {caminho}` · **{detalhes.get('operationId', '')}**")
    return spec["info"]["title"], spec["servers"][0]["url"], operacoes


# ------------------------------------------------------------------
# 4. Conversa, pergunta e ferramentas usadas
# ------------------------------------------------------------------
def nova_conversa():
    st.session_state.conversation_id = criar_clientes().conversations.create().id
    st.session_state.messages = []


def perguntar(texto):
    return criar_clientes().responses.create(
        conversation=st.session_state.conversation_id,
        extra_body={"agent": {"name": AGENT_NAME, "type": "agent_reference"}},
        input=texto,
    )


def ferramentas_usadas(response):
    # Tudo o que não é mensagem na saída é uma etapa do agente (ex.: chamada da ferramenta OpenAPI)
    return [f"{item.type} {getattr(item, 'name', '') or ''}".strip() for item in response.output if item.type != "message"]


# ------------------------------------------------------------------
# 5. Interface
# ------------------------------------------------------------------
st.title("🌦️ Agente do clima")
st.caption("Foundry Agent Service + ferramenta OpenAPI: o agente lê a especificação e chama a API wttr.in sozinho.")

if not FOUNDRY_PROJECT_ENDPOINT or not MODEL_DEPLOYMENT_NAME:
    st.error("Configure FOUNDRY_PROJECT_ENDPOINT e MODEL_DEPLOYMENT_NAME no arquivo `.env`.")
    st.stop()

if "conversation_id" not in st.session_state:
    try:
        nova_conversa()
    except ClientAuthenticationError:
        st.error("Falha no login da Azure. Execute `az login` no terminal e recarregue a página.")
        st.stop()
    except (HttpResponseError, APIError) as erro:
        st.error(f"Erro ao criar a conversa no Foundry: {erro}")
        st.stop()

with st.sidebar:
    st.header("🔌 Ferramenta OpenAPI")
    titulo, servidor, operacoes = ler_operacoes()
    st.markdown(f"**{titulo}**  \nServidor: `{servidor}`")
    for operacao in operacoes:
        st.markdown(f"- {operacao}")
    st.divider()
    st.caption(f"Agente: `{AGENT_NAME}`  \nConversa: `{st.session_state.conversation_id}`")
    if st.button("🗑️ Nova conversa"):
        nova_conversa()
        st.rerun()

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])
        if message.get("etapas"):
            st.caption("🔧 " + " → ".join(message["etapas"]))

if prompt := st.chat_input("Ex.: Como está o tempo em Londres hoje?"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Consultando a API do clima..."):
            try:
                response = perguntar(prompt)
            except ClientAuthenticationError:
                st.error("Falha no login da Azure. Execute `az login` no terminal.")
            except (HttpResponseError, APIError) as erro:
                st.error(f"Erro ao consultar o agente: {erro}")
            else:
                etapas = ferramentas_usadas(response)
                st.markdown(response.output_text)
                if etapas:
                    st.caption("🔧 " + " → ".join(etapas))
                st.session_state.messages.append({"role": "assistant", "content": response.output_text, "etapas": etapas})
